In [2]:

from pathlib import Path
import pandas as pd

source_file = Path("data/raw/acs_2023/ACSDT5Y2023.B17001-Data.csv")
output_folder = Path("data/processed/acs_2023")
output_folder.mkdir(parents=True, exist_ok=True)

In [8]:

raw = pd.read_csv(source_file, dtype={"GEO_ID": "string"})

print("Rows and columns:", raw.shape)
print(raw.head())
print(raw.columns[:8].tolist())

Rows and columns: (1334, 121)
                 GEO_ID                                        NAME  \
0             Geography                        Geographic Area Name   
1        0500000US17031                       Cook County, Illinois   
2  1400000US17031010100     Census Tract 101; Cook County; Illinois   
3  1400000US17031010201  Census Tract 102.01; Cook County; Illinois   
4  1400000US17031010202  Census Tract 102.02; Cook County; Illinois   

        B17001_001E              B17001_001M  \
0  Estimate!!Total:  Margin of Error!!Total:   
1           5101747                     1525   
2              3627                      621   
3              7578                      918   
4              2288                      417   

                                         B17001_002E  \
0  Estimate!!Total:!!Income in the past 12 months...   
1                                             680528   
2                                                508   
3                             

In [9]:
tracts = raw[
    raw["GEO_ID"].str.startswith("1400000US17031", na=False)
].copy()

print("Cook County tract rows:", len(tracts))

Cook County tract rows: 1332


In [10]:

poverty = tracts[
    [
        "GEO_ID",
        "NAME",
        "B17001_001E",
        "B17001_001M",
        "B17001_002E",
        "B17001_002M",
    ]
].copy()

poverty = poverty.rename(columns={
    "GEO_ID": "census_geoid",
    "NAME": "tract_name",
    "B17001_001E": "poverty_universe",
    "B17001_001M": "poverty_universe_moe",
    "B17001_002E": "below_poverty",
    "B17001_002M": "below_poverty_moe",
})

In [11]:
numeric_columns = [
    "poverty_universe",
    "poverty_universe_moe",
    "below_poverty",
    "below_poverty_moe",
]

for column in numeric_columns:
    poverty[column] = pd.to_numeric(poverty[column], errors="coerce")

poverty["GEOID"] = poverty["census_geoid"].str.extract(r"US(\d+)$")

In [12]:
poverty["poverty_rate_pct"] = (
    poverty["below_poverty"] / poverty["poverty_universe"] * 100
)

# Leave the rate missing when the denominator is zero or unavailable.
poverty.loc[poverty["poverty_universe"].le(0), "poverty_rate_pct"] = pd.NA

In [13]:
print("Duplicate tract GEOIDs:", poverty["GEOID"].duplicated().sum())
print("\nMissing values:")
print(poverty.isna().sum())

print("\nPoverty count exceeds the poverty-status total:",
      (poverty["below_poverty"] > poverty["poverty_universe"]).sum())

print("\nPoverty rates outside 0–100:",
      ((poverty["poverty_rate_pct"] < 0) |
       (poverty["poverty_rate_pct"] > 100)).sum())

poverty.head()

Duplicate tract GEOIDs: 0

Missing values:
census_geoid            0
tract_name              0
poverty_universe        0
poverty_universe_moe    0
below_poverty           0
below_poverty_moe       0
GEOID                   0
poverty_rate_pct        4
dtype: int64

Poverty count exceeds the poverty-status total: 0

Poverty rates outside 0–100: 0


,census_geoid,tract_name,poverty_universe,poverty_universe_moe,below_poverty,below_poverty_moe,GEOID,poverty_rate_pct
2,1400000US17031010100,Census Tract 101; Cook County; Illinois,3627,621,508,223,17031010100,14.006066
3,1400000US17031010201,Census Tract 102.01; Cook County; Illinois,7578,918,1892,745,17031010201,24.967010
4,1400000US17031010202,Census Tract 102.02; Cook County; Illinois,2288,417,524,101,17031010202,22.902098
5,1400000US17031010300,Census Tract 103; Cook County; Illinois,5384,907,431,226,17031010300,8.005201
6,1400000US17031010400,Census Tract 104; Cook County; Illinois,2865,526,463,211,17031010400,16.160558


In [15]:
poverty.loc[
    poverty["poverty_rate_pct"].isna(),
    ["GEOID", "tract_name", "poverty_universe", "below_poverty"]
]

,GEOID,tract_name,poverty_universe,below_poverty
413,17031381700,Census Tract 3817; Cook County; Illinois,0,0
1331,17031980000,Census Tract 9800; Cook County; Illinois,0,0
1332,17031980100,Census Tract 9801; Cook County; Illinois,0,0
1333,17031990000,Census Tract 9900; Cook County; Illinois,0,0


In [16]:
poverty.to_csv(output_folder / "poverty_tracts_clean.csv", index=False)

missing_summary = poverty.isna().sum().rename("missing_count").to_frame()
missing_summary.to_csv(output_folder / "poverty_missing_values.csv")

print("Saved cleaned tract data and missing-value summary.")

Saved cleaned tract data and missing-value summary.
